# WriteLens: fine-tune a detector with QLoRA, then evaluate it

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**

Trains Qwen2.5-0.5B with 4-bit QLoRA on HC3 (evaluation questions held out), then scores the same 200 + 200 answers as the GPT-2 heuristic for a fair comparison.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to T4 GPU first'

In [ ]:
!git clone -q https://github.com/sandhya-bdb/WriteLens.git
%cd WriteLens
!pip -q install -r requirements-train.txt

## 1. Fine-tune (about 15 to 25 minutes on a T4)

In [ ]:
!python training/finetune_detector.py --per-class 2000 --epochs 1

## 2. Evaluate on the held-out answers

In [ ]:
!python scripts/evaluate.py --model finetuned --n 200

In [ ]:
print(open('eval/RESULTS_finetuned.md').read())
print(open('models/writelens-qwen-lora/training_metrics.json').read())

## 3. Download results and the adapter

In [ ]:
!cd models && zip -qr ../writelens-qwen-lora.zip writelens-qwen-lora -x '*/checkpoints/*'
from google.colab import files
for f in ['eval/results_finetuned.json', 'eval/RESULTS_finetuned.md',
          'models/writelens-qwen-lora/training_metrics.json', 'writelens-qwen-lora.zip']:
    files.download(f)